In [1]:

# ============================================================
# TIME-SERIES CONTINUITY AND MISSING-DATE ANALYSIS
# Project: Predictive Forecasting of Care Load & Placement Demand
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Project directory and dataset loading
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)

DATASET_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "cleaned_care_data.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATASET_PATH}"
    )

df = pd.read_csv(DATASET_PATH)

DATE_COLUMN = "date"
TARGET_VARIABLE = "children_in_hhs_care"

required_columns = [DATE_COLUMN, TARGET_VARIABLE]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Required columns missing: {missing_columns}"
    )

# ------------------------------------------------------------
# Data preparation and chronological sorting
# ------------------------------------------------------------

df[DATE_COLUMN] = pd.to_datetime(
    df[DATE_COLUMN],
    errors="coerce"
)

invalid_date_count = int(df[DATE_COLUMN].isna().sum())

df = df.dropna(subset=[DATE_COLUMN]).copy()

for col in df.columns:
    if col != DATE_COLUMN:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(
                ",", "", regex=False
            ),
            errors="coerce"
        )

df = df.sort_values(DATE_COLUMN).reset_index(drop=True)

if df[DATE_COLUMN].duplicated().any():
    duplicate_dates = df.loc[
        df[DATE_COLUMN].duplicated(keep=False),
        DATE_COLUMN
    ].tolist()

    raise ValueError(
        f"Duplicate dates detected: {duplicate_dates[:20]}. "
        "Resolve duplicates before continuing."
    )

minimum_date = df[DATE_COLUMN].min()
maximum_date = df[DATE_COLUMN].max()

print("=" * 65)
print("DATASET OVERVIEW")
print("=" * 65)

print("Project directory:", PROJECT_ROOT)
print("Dataset:", DATASET_PATH)
print("Dataset shape:", df.shape)
print("Invalid dates removed:", invalid_date_count)
print("Minimum date:", minimum_date.date())
print("Maximum date:", maximum_date.date())
print("Observed records:", len(df))

# ------------------------------------------------------------
# Date differences and reporting intervals
# ------------------------------------------------------------

df["Date_Difference_Days"] = (
    df[DATE_COLUMN].diff().dt.days
)

date_differences = (
    df["Date_Difference_Days"].dropna()
)

interval_counts = (
    date_differences.value_counts().sort_index()
)

reporting_frequency_df = pd.DataFrame({
    "Reporting_Interval_Days": interval_counts.index,
    "Frequency": interval_counts.values
})

if not reporting_frequency_df.empty:

    reporting_frequency_df["Percentage"] = (
        reporting_frequency_df["Frequency"] /
        reporting_frequency_df["Frequency"].sum() * 100
    )

    reporting_frequency_df["Interval_Description"] = (
        reporting_frequency_df["Reporting_Interval_Days"].apply(
            lambda x:
            "Daily" if x == 1
            else "Weekly" if x == 7
            else "Monthly-like" if 28 <= x <= 31
            else "Other interval"
        )
    )

else:

    reporting_frequency_df["Percentage"] = pd.Series(dtype=float)
    reporting_frequency_df["Interval_Description"] = pd.Series(dtype=str)

reporting_frequency_df.to_csv(
    OUTPUT_DIR / "reporting_frequency_analysis.csv",
    index=False
)

print("\nREPORTING INTERVAL STATISTICS")

if not date_differences.empty:
    print(date_differences.describe())
else:
    print("Insufficient dates for interval analysis.")

# ------------------------------------------------------------
# Reporting frequency determination
# ------------------------------------------------------------

if not interval_counts.empty:

    most_common_interval = int(interval_counts.idxmax())

    if most_common_interval == 1:
        observed_frequency = "Daily"
    elif most_common_interval == 7:
        observed_frequency = "Weekly"
    elif 28 <= most_common_interval <= 31:
        observed_frequency = "Monthly-like"
    else:
        observed_frequency = (
            f"Most common interval: {most_common_interval} days"
        )

else:

    most_common_interval = np.nan
    observed_frequency = "Undetermined"

print("\nObserved reporting frequency:", observed_frequency)

# ------------------------------------------------------------
# Complete daily calendar and missing-date identification
# ------------------------------------------------------------

complete_calendar = pd.date_range(
    start=minimum_date,
    end=maximum_date,
    freq="D"
)

observed_dates = pd.DatetimeIndex(df[DATE_COLUMN])

missing_dates = complete_calendar.difference(
    observed_dates
)

missing_date_df = pd.DataFrame({
    "Date": missing_dates
})

# ------------------------------------------------------------
# Consecutive missing-date periods
# ------------------------------------------------------------

if not missing_date_df.empty:

    missing_date_df["Gap_Group"] = (
        missing_date_df["Date"]
        .diff()
        .dt.days
        .ne(1)
        .cumsum()
    )

    gap_summary = (
        missing_date_df.groupby("Gap_Group")
        .agg(
            Gap_Start=("Date", "min"),
            Gap_End=("Date", "max"),
            Missing_Days=("Date", "count")
        )
        .reset_index()
    )

    missing_date_df = missing_date_df.merge(
        gap_summary,
        on="Gap_Group",
        how="left"
    )

    observed_series = pd.Series(
        observed_dates.sort_values()
    )

    missing_date_df["Previous_Observed_Date"] = (
        missing_date_df["Date"].apply(
            lambda d: observed_series[
                observed_series < d
            ].max()
        )
    )

    missing_date_df["Next_Observed_Date"] = (
        missing_date_df["Date"].apply(
            lambda d: observed_series[
                observed_series > d
            ].min()
        )
    )

else:

    gap_summary = pd.DataFrame(columns=[
        "Gap_Group",
        "Gap_Start",
        "Gap_End",
        "Missing_Days"
    ])

    for col in [
        "Gap_Group",
        "Gap_Start",
        "Gap_End",
        "Missing_Days",
        "Previous_Observed_Date",
        "Next_Observed_Date"
    ]:
        missing_date_df[col] = pd.Series(dtype="object")

missing_date_df["Reporting_Status"] = (
    "No observation recorded; reporting status unconfirmed"
)

missing_date_df.to_csv(
    OUTPUT_DIR / "missing_date_analysis.csv",
    index=False
)

# ------------------------------------------------------------
# Reporting patterns by weekday and month
# ------------------------------------------------------------

df["Weekday"] = df[DATE_COLUMN].dt.day_name()
df["Month"] = df[DATE_COLUMN].dt.month

weekday_reporting = (
    df["Weekday"].value_counts()
    .reindex(
        [
            "Monday", "Tuesday", "Wednesday",
            "Thursday", "Friday", "Saturday", "Sunday"
        ],
        fill_value=0
    )
)

monthly_reporting = (
    df.groupby(
        df[DATE_COLUMN].dt.to_period("M")
    ).size()
)

print("\nREPORTING BY WEEKDAY")
print(weekday_reporting)

print("\nREPORTING BY MONTH")
print(monthly_reporting)

# ------------------------------------------------------------
# Complete daily calendar index
# ------------------------------------------------------------

original_indexed = (
    df.set_index(DATE_COLUMN)
    .drop(columns=[
        "Date_Difference_Days",
        "Weekday",
        "Month"
    ])
)

daily_df = original_indexed.reindex(
    complete_calendar
)

daily_df.index.name = DATE_COLUMN

daily_df["Observed_Date"] = (
    daily_df.index.isin(observed_dates).astype(int)
)

daily_df["Estimated_Date"] = (
    1 - daily_df["Observed_Date"]
)

daily_df["Reporting_Status"] = np.where(
    daily_df["Observed_Date"].eq(1),
    "Observed",
    "Missing observation"
)

# Preserve missing observations as NaN
daily_df.to_csv(
    OUTPUT_DIR / "time_series_continuity.csv"
)

# ------------------------------------------------------------
# Original and complete date-index comparison
# ------------------------------------------------------------

total_calendar_days = len(complete_calendar)
observed_count = len(observed_dates)
missing_count = len(missing_dates)

index_comparison = pd.DataFrame({
    "Metric": [
        "Original observed dates",
        "Complete calendar dates",
        "Missing calendar dates",
        "Observed date percentage",
        "Missing date percentage",
        "Minimum date",
        "Maximum date"
    ],
    "Value": [
        observed_count,
        total_calendar_days,
        missing_count,
        round(observed_count / total_calendar_days * 100, 2),
        round(missing_count / total_calendar_days * 100, 2),
        str(minimum_date.date()),
        str(maximum_date.date())
    ]
})

print("\nDATE INDEX COMPARISON")
print(index_comparison.to_string(index=False))

# ------------------------------------------------------------
# Missing-date reporting investigation
# ------------------------------------------------------------

print("\nMISSING-DATE ANALYSIS")
print("Total calendar days:", total_calendar_days)
print("Observed dates:", observed_count)
print("Missing dates:", missing_count)

print(
    "Reporting coverage:",
    round(observed_count / total_calendar_days * 100, 2),
    "%"
)

print("\nCONSECUTIVE MISSING-DATE PERIODS")

if not gap_summary.empty:
    print(gap_summary.to_string(index=False))
else:
    print("No missing calendar dates detected.")

print("\nREPORTING INTERPRETATION")
print(
    "Missing calendar dates indicate that no observation exists "
    "for those dates in the supplied dataset. They do not prove "
    "that reporting stopped or that custody values were zero."
)

# ------------------------------------------------------------
# Missing-value treatment alternatives
# ------------------------------------------------------------

target_series = daily_df[TARGET_VARIABLE].copy()

# Original observed values
observed_only = target_series.copy()

# Forward fill
forward_fill = target_series.ffill()

# Backward fill
backward_fill = target_series.bfill()

# Time interpolation
time_interpolation = target_series.interpolate(
    method="time",
    limit_area="inside"
)

# Seven-day rolling mean for diagnostic comparison
rolling_mean = target_series.copy()

rolling_values = target_series.rolling(
    window=7,
    min_periods=1
).mean()

rolling_mean.loc[target_series.isna()] = (
    rolling_values.loc[target_series.isna()]
)

treatment_methods = {
    "Observed_Only": observed_only,
    "Forward_Fill": forward_fill,
    "Backward_Fill": backward_fill,
    "Time_Interpolation": time_interpolation,
    "Rolling_7_Day_Mean": rolling_mean
}

# ------------------------------------------------------------
# Imputation comparison and forecast sensitivity
# ------------------------------------------------------------

forecast_horizons = [1, 7, 14, 30]

comparison_results = []

for method_name, series in treatment_methods.items():

    valid_values = series.dropna()

    result = {
        "Treatment_Method": method_name,
        "Observed_Values": int(target_series.notna().sum()),
        "Available_Values": int(series.notna().sum()),
        "Estimated_Values": int(
            series.notna().sum() -
            target_series.notna().sum()
        ),
        "Missing_Values_Remaining": int(series.isna().sum()),
        "Mean": series.mean(),
        "Standard_Deviation": series.std(),
        "Minimum": series.min(),
        "Maximum": series.max(),
        "Last_Available_Value": (
            valid_values.iloc[-1]
            if not valid_values.empty else np.nan
        )
    }

    last_value = (
        valid_values.iloc[-1]
        if not valid_values.empty else np.nan
    )

    for horizon in forecast_horizons:

        result[f"Naive_Forecast_{horizon}_Day"] = last_value

    recent = series.dropna().tail(30)

    if len(recent) >= 2:

        valid_positions = np.flatnonzero(
            series.notna().to_numpy()
        )

        recent_positions = valid_positions[-30:]

        slope, intercept = np.polyfit(
            recent_positions,
            recent.to_numpy(),
            1
        )

        for horizon in forecast_horizons:

            future_position = len(series) - 1 + horizon

            result[f"Trend_Forecast_{horizon}_Day"] = (
                slope * future_position + intercept
            )

    else:

        for horizon in forecast_horizons:
            result[f"Trend_Forecast_{horizon}_Day"] = np.nan

    comparison_results.append(result)

imputation_comparison_df = pd.DataFrame(
    comparison_results
)

imputation_comparison_df.to_csv(
    OUTPUT_DIR / "imputation_comparison.csv",
    index=False
)

# ------------------------------------------------------------
# Forecast sensitivity summary
# ------------------------------------------------------------

forecast_sensitivity = imputation_comparison_df[
    [
        "Treatment_Method",
        "Naive_Forecast_1_Day",
        "Naive_Forecast_7_Day",
        "Naive_Forecast_14_Day",
        "Naive_Forecast_30_Day",
        "Trend_Forecast_7_Day",
        "Trend_Forecast_14_Day",
        "Trend_Forecast_30_Day"
    ]
]

print("\nFORECAST SENSITIVITY COMPARISON")
print(forecast_sensitivity.to_string(index=False))

# ------------------------------------------------------------
# Final continuity validation
# ------------------------------------------------------------

assert daily_df.index.is_monotonic_increasing, (
    "Calendar index is not chronological."
)

assert daily_df.index.is_unique, (
    "Calendar index contains duplicate dates."
)

assert len(daily_df) == total_calendar_days, (
    "Calendar length validation failed."
)

assert int(daily_df["Observed_Date"].sum()) == observed_count, (
    "Observed-date indicator validation failed."
)

assert daily_df.index.to_series().diff().dropna().eq(
    pd.Timedelta(days=1)
).all(), "Daily calendar continuity validation failed."

# Compare original and reconstructed values using NumPy arrays
original_values = (
    original_indexed[TARGET_VARIABLE].to_numpy(dtype=float)
)

reconstructed_values = (
    daily_df.loc[
        original_indexed.index,
        TARGET_VARIABLE
    ].to_numpy(dtype=float)
)

assert np.allclose(
    original_values,
    reconstructed_values,
    equal_nan=True
), "Original observations were not preserved."

# Check that missing dates remain unobserved
assert daily_df.loc[
    missing_dates, "Observed_Date"
].eq(0).all()

# Check that missing target observations were not converted to zero
assert daily_df.loc[
    missing_dates, TARGET_VARIABLE
].isna().all()

# ------------------------------------------------------------
# Final deliverables verification
# ------------------------------------------------------------

deliverables = [
    "missing_date_analysis.csv",
    "reporting_frequency_analysis.csv",
    "time_series_continuity.csv",
    "imputation_comparison.csv"
]

print("\n" + "=" * 65)
print("TIME-SERIES CONTINUITY ANALYSIS COMPLETED")
print("=" * 65)

print("Observed reporting frequency:", observed_frequency)
print("Calendar index frequency: Daily")
print("Observed records:", observed_count)
print("Missing calendar dates:", missing_count)

print("\nMETHODOLOGICAL VALIDATION")
print("Missing dates converted to zero: NO")
print("Custody stock automatically interpolated: NO")
print("Original observations preserved: YES")
print("Observed-date indicators maintained: YES")
print("Daily calendar continuity validated: YES")
print("Forecast sensitivity alternatives compared: YES")

print("\nGENERATED DELIVERABLES")

for filename in deliverables:

    file_path = OUTPUT_DIR / filename

    assert file_path.exists(), (
        f"Deliverable missing: {filename}"
    )

    print(
        f"{filename} | "
        f"{file_path.stat().st_size:,} bytes"
    )

print("\nOutput directory:", OUTPUT_DIR.resolve())

print("\nALL VALIDATION CHECKS PASSED.")


DATASET OVERVIEW
Project directory: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand
Dataset: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\data\processed\cleaned_care_data.csv
Dataset shape: (720, 6)
Invalid dates removed: 0
Minimum date: 2023-01-12
Maximum date: 2025-12-21
Observed records: 720

REPORTING INTERVAL STATISTICS
count    719.000000
mean       1.493741
std        1.001198
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max       10.000000
Name: Date_Difference_Days, dtype: float64

Observed reporting frequency: Daily

REPORTING BY WEEKDAY
Weekday
Monday       145
Tuesday      149
Wednesday    147
Thursday     147
Friday         2
Saturday       0
Sunday       130
Name: count, dtype: int64

REPORTING BY MONTH
date
2023-01     8
2023-02    19
2023-03    19
2023-04    21
2023-05    22
2023-06    20
2023-07    21
2023-08    20
2023-09    19
2023-10    22
2023-11    20
2023-12    19
2024-01    22
2024-02   